# PulseCam – training on Colab (free GPU)
Runtime ▸ Change runtime type ▸ GPU. Checkpoints go to Google Drive so a disconnect loses nothing.
**Research prototype, not a medical device.** Never put volunteer videos in a public place.

## 1. Drive + code

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os, zipfile
BASE = '/content/drive/MyDrive/PulseCam'
os.makedirs(BASE + '/runs', exist_ok=True)
# code comes from PulseCam_code.zip that you uploaded to MyDrive/PulseCam/  (no GitHub push needed)
if not os.path.exists('/content/PulseCam'):
    zipfile.ZipFile(BASE + '/PulseCam_code.zip').extractall('/content/PulseCam')
%cd /content/PulseCam
!ls

## 2. rPPG-Toolbox + our patch (own-recordings loader, fine-tune hook)

In [ ]:
%cd /content/PulseCam
!test -d rPPG-Toolbox || git clone --depth 1 https://github.com/ubicomplab/rPPG-Toolbox.git
%cd /content/PulseCam/rPPG-Toolbox
# Colab ships its own torch/numpy; the toolbox pins Python 3.8 + torch 2.1 (setup.sh fails here). Install only the extras, unpinned.
# Colab has 3 clashing OpenCV wheels (contrib 4.14 + python/headless 5.0) -> heap corruption in preprocessing workers; keep exactly one
!pip uninstall -y -q opencv-python opencv-contrib-python opencv-python-headless
!pip install -q opencv-python-headless==4.10.0.84 yacs tensorboardX mat73 neurokit2 thop timm scikit-image h5py
# mamba_ssm (PhysMamba only) can't be built on Colab: add an empty stub so the toolbox still imports
import os, site
d = os.path.join(site.getsitepackages()[0], 'mamba_ssm'); os.makedirs(d, exist_ok=True)
open(d + '/__init__.py', 'w').write('class Mamba:' + chr(10) + '    pass' + chr(10))
!python ../toolbox_patch/install.py .
!ln -sfn /content/drive/MyDrive/PulseCam/runs runs   # checkpoints -> Drive
!mkdir -p ../results/logs

## 3. Data (UBFC-rPPG)
Google blocks scripted downloads of UBFC ("too many users"). Two ways that work from your own account:
1. **Shortcut (no copying):** open the [UBFC Drive folder](https://drive.google.com/drive/folders/1o0XU4gTIo46YfwaWjIgbtCncc-oF44Xk), right-click `DATASET_2` > **Organize > Add shortcut** > My Drive. Set `UBFC_SRC` below to that shortcut.
2. **Upload:** download a few `subjectN` folders in your browser and upload them to `MyDrive/PulseCam/data/UBFC-rPPG/` (each folder holds `vid.avi` + `ground_truth.txt`). Set `UBFC_SRC` to that folder.

PURE: email the authors. SCAMPS: direct download. Faces of volunteers never go to a public place.

In [ ]:
import glob, os, re, shutil
UBFC_SRC = '/content/drive/MyDrive/DATASET_2'   # <- shortcut or upload folder; must contain subject*/vid.avi
UBFC_DST = '/content/data/UBFC-rPPG'
MAX_SUBJECTS = 2                            # e.g. 5 for a quick test run
subs = sorted(glob.glob(UBFC_SRC + '/subject*'), key=lambda s: int(re.findall(r'\d+', s)[-1]))[:MAX_SUBJECTS]
os.makedirs(UBFC_DST, exist_ok=True)
for s in subs:
    d = os.path.join(UBFC_DST, os.path.basename(s))
    if not os.path.exists(d + '/vid.avi'):
        shutil.copytree(s, d, dirs_exist_ok=True)
ok = [d for d in sorted(glob.glob(UBFC_DST + '/subject*')) if os.path.exists(d + '/vid.avi') and os.path.exists(d + '/ground_truth.txt')]
print(len(ok), 'subjects ready:', [os.path.basename(d) for d in ok])
assert ok, 'no subject*/vid.avi + ground_truth.txt found - check UBFC_SRC'

# point the toolbox UBFC test configs at the local copy (first run preprocesses)
CACHE = '/content/data/PreprocessedData'
cfgs = ['UBFC-rPPG_UNSUPERVISED'] + [f'PURE_UBFC-rPPG_{m}_BASIC' for m in ['PHYSNET', 'TSCAN', 'DEEPPHYS']]
def set_key(t, key, val):  # replace the quoted value after `key:`
    return re.sub(r'(' + key + r':\s*)"[^"]*"', lambda m: m.group(1) + '"' + val + '"', t)

for c in cfgs:
    f = f'/content/PulseCam/rPPG-Toolbox/configs/infer_configs/{c}.yaml'
    t = open(f).read()
    t = set_key(t, 'DATA_PATH', UBFC_DST)
    t = set_key(t, 'CACHED_PATH', CACHE)
    t = re.sub(r'DO_PREPROCESS:\s*False', 'DO_PREPROCESS: True', t)
    open(f, 'w').write(t)
print('patched', cfgs)

## 4. E1 + E2 - baselines vs deep models on UBFC-rPPG
Uses our own scorer (`eval/ubfc_eval.py`): the toolbox's `main.py` gave bogus numbers here (cached preprocessing scored only 2 samples). Needs mediapipe and exactly one OpenCV wheel.


In [ ]:
# mediapipe pulls a 2nd OpenCV wheel that crashes Python (segfault) -> keep exactly one
!pip install -q mediapipe scikit-image scikit-learn
!pip uninstall -y -q opencv-contrib-python opencv-python opencv-python-headless
!pip install -q opencv-python-headless==4.10.0.84
%cd /content/PulseCam
!python eval/ubfc_eval.py --root /content/data/UBFC-rPPG --max 10 2>&1 | tee results/logs/E1_E2_ubfc_eval.log
# tables -> results/tables/E1_E2_ubfc*.csv, plots -> results/plots/ ; copy them to Drive before the session ends:
!mkdir -p /content/drive/MyDrive/PulseCam/results_out && cp -r results/tables results/plots /content/drive/MyDrive/PulseCam/results_out/


Also train your own on UBFC subjects (copy a train config, set TRAIN/VALID/TEST DATASET to UBFC-rPPG with BEGIN/END subject splits) and `tee` to `results/logs/E1_trained_<model>.log`.

## 5. E2 - cross-dataset (train PURE or SCAMPS, test UBFC-rPPG)
Already in the table above: `TSCAN_PURE` and `TSCAN_SCAMPS` are the toolbox's released checkpoints tested on UBFC. Only run the cell below to *train your own* (needs PURE or SCAMPS data; may hit the toolbox cache problem - check the sample count in the log).


In [ ]:
!python main.py --config_file configs/train_configs/PURE_PURE_UBFC-rPPG_TSCAN_BASIC.yaml 2>&1 | tee ../results/logs/E2_PURE_to_UBFC_tscan.log
# if PURE access is late:
# !python main.py --config_file configs/train_configs/SCAMPS_SCAMPS_UBFC-rPPG_TSCAN_BASIC.yaml 2>&1 | tee ../results/logs/E2_SCAMPS_to_UBFC_tscan.log

## 6. Fine-tune on Indian volunteers (E4)
Copy `data/own/videos` to Colab (private Drive), fix the paths in the yaml, then:

In [ ]:
import os; os.environ['PULSECAM_INIT'] = 'final_model_release/PURE_TSCAN.pth'
!python main.py --config_file ../configs/train/PulseCam_finetune_TSCAN.yaml 2>&1 | tee ../results/logs/E4_finetune.log

## 7. Collect results + save the best weights

In [ ]:
!python ../eval/parse_logs.py
import glob, shutil
ck = sorted(glob.glob('runs/exp/*/PreTrainedModels/*Epoch*.pth') or glob.glob('runs/**/*Epoch*.pth', recursive=True))
print(ck[-3:])
# shutil.copy(ck[-1], '../models/best_tscan.pth')  # pick the checkpoint that beat the baselines; the demo then uses it